# Forecasting via Lag-Feature Attention
Thomas Keenan

**Table of Contents**


#### Imports and Modelling framework

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor

sns.set_style("whitegrid")
seed = 40304451
np.random.seed(seed)


In [13]:
tickers = ["XLK", "XLP", "XLV", "XLF", "XLE", "XLI"]
# Download each ticker
all_data = []
for ticker in tickers:
    stock = yf.Ticker(ticker)
    hist = stock.history(start="2010-01-01", end="2024-12-31", auto_adjust=False)
    hist['ticker'] = ticker
    hist = hist.reset_index()
    all_data.append(hist)
    
# Combine into one big df
prices_df = pd.concat(all_data, ignore_index=True)

# Change date column type
prices_df['Date'] = pd.to_datetime(prices_df['Date']).dt.tz_localize(None)

# Format column names for mltester
prices_df = prices_df.rename(columns={
    'Date': 'date',
    'Open': 'open', 
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'Adj Close': 'adj_close'
})

# Replace 0s with NaN in important cols
predictive_cols = ['open', 'high', 'low', 'close', 'volume']
prices_df[predictive_cols] = prices_df[predictive_cols].replace(0, np.nan)

# Set the date as the index and sort
prices_df = prices_df.set_index('date')
prices_df = prices_df.sort_values(['ticker', 'date'])

Missingness check:

In [15]:
print(prices_df.isna().sum())

open             0
high             0
low              0
close            0
adj_close        0
volume           0
Dividends        0
Stock Splits     0
Capital Gains    0
ticker           0
dtype: int64


There should be no missing values (as of December 2025)

### Forecaster Class:
The Forecaster class was developed as part of my original Forecasting H-Day cumulative log returns project, but has been streamlined given this study's scope.

It can take a JSON-style 'config' such as:
````python
configs = { 
    'task_1_model': {
        'n_lags': 15,
        'parkinson_vol_windows': (5, 10),
        'alpha_grid': (10.0, 50.0, 100.0)
    }
}
````
Which will test a Ridge model (the default model type) with strong L2 regularization using 15 day lagged returns (defaulted to 5) and parkinson volatility windows.

In [81]:
class Forecaster:
    '''Modelling class - input feature details and model architures. '''
    def __init__(
        self,
        config=None,
        random_state=seed,
    ):
        # Set defaults:
        
        # 5 lags was adopted in previous study
        self.n_lags = 5

        self.mom_windows = ()
        self.vol_window = None
        self.sma_windows = ()
        self.ema_windows = ()
        self.rsi_window = None
        self.parkinson_vol_windows = ()
        self.use_parkinson_vol_regime = False
        
        self.model_type = 'ridge'
        self.alpha_grid = (0.01, 0.1, 1.0, 10.0)
        self.cv_splits = 3
        self.min_train_points = 200
        self.random_state = random_state
                             
        # Override features with config details
        if isinstance(config, dict):
            for k, v in config.items():
                if hasattr(self, k):
                    setattr(self, k, v)

        # Convert types after config is applied
        self.n_lags = int(self.n_lags) if self.n_lags else 0
        self.parkinson_vol_windows = tuple(int(w) for w in self.parkinson_vol_windows) if self.parkinson_vol_windows else None
        
        self.model_type = str(self.model_type).lower()
        self.alpha_grid = tuple(float(a) for a in self.alpha_grid)
        self.cv_splits = int(self.cv_splits)
        self.min_train_points = int(self.min_train_points)
        self.random_state = int(self.random_state)
    
        self.pipe_ = None
        self.best_alpha_ = None
        self.fitted_ = False

    @staticmethod
    def _close_series(X: pd.DataFrame) -> pd.Series:
        return X["Close"] if "Close" in X.columns else X.iloc[:, 0]

    @staticmethod
    def _log_returns(series: pd.Series) -> pd.Series:
        series = pd.Series(series).astype(float)
        return np.log(series / series.shift(1))

    @staticmethod
    def _finite_mean(y: pd.Series) -> float:
        """Return finite mean of y or 0.0 if none available."""
        yv = pd.Series(y).astype(float).replace([np.inf, -np.inf], np.nan).dropna()
        if len(yv) == 0:
            return 0.0
        m = float(yv.mean())
        return m if np.isfinite(m) else 0.0
    
    @staticmethod
    def _parkinson_volatility(high: pd.Series, low: pd.Series, window: int) -> pd.Series:
        high = pd.Series(high).astype(float)
        low = pd.Series(low).astype(float)
        
        hl_ratio = (high / low).replace(0, np.nan)
        squared_log = (np.log(hl_ratio) ** 2)
        
        parkinson_vol = np.sqrt(squared_log.rolling(window, min_periods=window).mean() / (4.0 * np.log(2)))
        
        return parkinson_vol

    def _make_features(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Build features based on config.
        """
        close = self._close_series(X).astype(float)
        lr = self._log_returns(close)
    
        feats = {}

        # Lags
        if self.n_lags > 0:
            for i in range(1, self.n_lags + 1):
                # Raw lags
                feats[f"lag{i}"] = lr.shift(i)
        
        # Parkinson Volatility (needs OHLC)
        if self.parkinson_vol_windows is not None:
            if all(col in X.columns for col in ["High", "Low"]):
                windows_to_use = list(self.parkinson_vol_windows)
                
                for w in windows_to_use:
                    feats[f"parkinson_vol_{w}"] = self._parkinson_volatility(
                        X["High"], X["Low"], w
                    ).shift(1)
   
        # Return empty DataFrame if no features
        if not feats:
            return pd.DataFrame(index=X.index)
    
        F = pd.DataFrame(feats, index=X.index).replace([np.inf, -np.inf], np.nan)
        F = F.dropna()
        return F

    def _get_model(self, alpha=None):
        if self.model_type == 'somethingelse':
            return Ridge(alpha=alpha or 1.0, random_state=self.random_state)
        
        else:
            # The default model for this work is Ridge
            return Ridge(alpha=alpha or 1.0, random_state=self.random_state)

    def fit(self, X_train: pd.DataFrame, y_train: pd.Series, meta=None):
        """
        Train with selected model architecture
        """
        F = self._make_features(X_train)

        if F.empty:
            mean_y = self._finite_mean(y_train)
            self.pipe_ = Pipeline([("model", DummyRegressor(strategy="constant", constant=mean_y))])
            self.pipe_.fit([[0.0]], [0.0])
            self.best_alpha_ = None
            self.fitted_ = True
            return self

        y = y_train.reindex(F.index)
        mask = y.replace([np.inf, -np.inf], np.nan).notna()
        F, y = F.loc[mask], y.loc[mask]

        if len(y) == 0:
            mean_y = self._finite_mean(y_train)
            self.pipe_ = Pipeline([("model", DummyRegressor(strategy="constant", constant=mean_y))])
            self.pipe_.fit([[0.0]], [0.0])
            self.best_alpha_ = None
            self.fitted_ = True
            return self

        if len(F) < self.min_train_points:
            mean_y = float(y.mean()) if np.isfinite(y.mean()) else 0.0
            self.pipe_ = Pipeline([("model", DummyRegressor(strategy="constant", constant=mean_y))])
            self.pipe_.fit([[0.0]], [0.0])
            self.best_alpha_ = None
            self.fitted_ = True
            return self

        # For linear models, tune alpha with scaling
        n_splits = min(self.cv_splits, max(2, len(F) // 200))
        tscv = TimeSeriesSplit(n_splits=n_splits)

        best_alpha, best_mse = None, np.inf
        for a in self.alpha_grid:
            mses = []
            for tr_idx, va_idx in tscv.split(F.values):
                X_tr, X_va = F.values[tr_idx], F.values[va_idx]
                y_tr, y_va = y.values[tr_idx], y.values[va_idx]
                
                model = self._get_model(alpha=a)
                pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
                pipe.fit(X_tr, y_tr)
                y_hat = pipe.predict(X_va)
                mses.append(mean_squared_error(y_va, y_hat))
            
            avg_mse = float(np.mean(mses)) if mses else np.inf
            if avg_mse < best_mse:
                best_mse, best_alpha = avg_mse, a

        if best_alpha is None:
            best_alpha = 1.0
        self.best_alpha_ = float(best_alpha)

        model = self._get_model(alpha=self.best_alpha_)
        self.pipe_ = Pipeline([("scaler", StandardScaler()), ("model", model)])
        self.pipe_.fit(F.values, y.values)
        self.fitted_ = True
        return self

    def predict(self, X: pd.DataFrame, meta=None) -> pd.Series:
        """
        Return predictions of next-H-day cumulative log returns.
        """
        F = self._make_features(X)
        if not self.fitted_ or self.pipe_ is None:
            idx = F.index if len(F) else X.index
            return pd.Series(0.0, index=idx, name="y_pred")
        if F.empty:
            return pd.Series(0.0, index=X.index, name="y_pred")
        y_hat = self.pipe_.predict(F.values)
        return pd.Series(y_hat, index=F.index, name="y_pred")

#### Helper Functions

In [84]:
def get_ticker_data(prices_df, ticker):
    """Extract single ticker and format for Forecaster class"""
    if prices_df.index.name == 'date':
        df = prices_df.reset_index()
    else:
        df = prices_df.copy()
    
    ticker_data = df[df['ticker'] == ticker].copy()
    if ticker_data.empty:
        raise ValueError(f"No data for {ticker}")
    
    ticker_data = ticker_data.drop(columns=['ticker'])
    if 'date' in ticker_data.columns:
        ticker_data = ticker_data.set_index('date')
    
    ticker_data.columns = [col.capitalize() for col in ticker_data.columns]
    ticker_data.index = pd.to_datetime(ticker_data.index)
    
    return ticker_data.sort_index()

def forward_log_return(close, horizon):
    """Target: log(Close[t+H] / Close[t])"""
    close = pd.Series(close).astype(float)
    return np.log(close.shift(-horizon) / close)

def compute_metrics(y_true, y_pred):
    """Compute DirAcc, MAE, RMSE"""
    df = pd.concat([y_true.rename("y"), y_pred.rename("yhat")], axis=1)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    
    if df.empty:
        return {"diracc": np.nan, "mae": np.nan, "rmse": np.nan}
    
    diracc = float((np.sign(df["y"]) == np.sign(df["yhat"])).mean())
    mae = float(np.abs(df["y"] - df["yhat"]).mean())
    rmse = float(np.sqrt(((df["y"] - df["yhat"]) ** 2).mean()))
    
    return {"diracc": diracc, "mae": mae, "rmse": rmse}

#### Walk Forward Validation
This method calls a forecasting model based on the configuration inputted, then evaluates it using walk-forward validation with an expanding training window. To respect the temporal characteristic of time series data the model is retained at each step on all historical data up to the test period, ensuring realistic out-of-sample predictions.

In [87]:
def walk_forward(prices, config, horizon=5, step=10):
    """
    Test a Forecasting configuration using walk-forward validation.
    """
    y_full = forward_log_return(prices["Close"], horizon=horizon)
    y_full.name = "y_true"
    test_dates = y_full.dropna().index
    
    all_preds = []
    
    for i in range(0, len(test_dates), step):
        block = test_dates[i : i + step]
        first_test = block[0]
        
        # Expanding window: train on everything before first test date
        X_train = prices.loc[: first_test - pd.Timedelta(days=1)]
        y_train = y_full.loc[: first_test - pd.Timedelta(days=1)]

        # Set a model based on the Forecaster configuration
        model = Forecaster(config=config)

        # Train
        model.fit(X_train, y_train, meta={"horizon": horizon})
        
        X_pred = prices.loc[: block[-1]]
        # Test
        y_hat = model.predict(X_pred, meta={"horizon": horizon})
        
        all_preds.append(y_hat.reindex(block).dropna())
    
    # Combine all predictions
    y_pred = pd.concat(all_preds) if all_preds else pd.Series(dtype=float, name="y_pred")
    y_true = y_full.reindex(y_pred.index)
    
    return y_true, y_pred

#### Run Forecasting Tests:

In [90]:
def run_evaluation(
    configs,
    prices_df,
    tickers,
    date_start,
    date_end,
    horizon,
    step,
    verbose=True,
    return_predictions=False
):
    """
    Evaluate multiple forecasting configurations across tickers using walk-forward validation.
    For each configuration and ticker combination, performs walk-forward validation and computes directional accuracy, MAE, and RMSE metrics.
    """
    results = []
    
    for config_name, config_params in configs.items():
        if verbose:
            print(f"\n{'_'*60}")
            print(f"Config: {config_name}")
            print(f"Config: {config_params}\n")
        
        for ticker in tickers:
            ticker_prices = get_ticker_data(prices_df, ticker)
            ticker_prices = ticker_prices.loc[date_start:date_end]
            
            y_true, y_pred = walk_forward(
                ticker_prices, 
                config_params, 
                horizon=horizon, 
                step=step
            )
            
            metrics = compute_metrics(y_true, y_pred)
            
            if return_predictions:
                for date, true_val, pred_val in zip(y_true.index, y_true.values, y_pred.values):
                    results.append({
                        'config': config_name,
                        'ticker': ticker,
                        'date': date,
                        'y_true': true_val,
                        'y_pred': pred_val,
                        'diracc': metrics['diracc'],
                        'mae': metrics['mae'],
                        'rmse': metrics['rmse'],
                    })
            else:
                results.append({
                    'config': config_name,
                    'ticker': ticker,
                    'diracc': metrics['diracc'],
                    'mae': metrics['mae'],
                    'rmse': metrics['rmse'],
                })
            
            if verbose:
                # Log individual ticker results
                print(f"{ticker}: DirAcc={metrics['diracc']:.4f}, "
                      f"MAE={metrics['mae']:.6f}, RMSE={metrics['rmse']:.6f}, ")
    
    results_df = pd.DataFrame(results)
    if verbose:
        print("\nTest Results:")
        print(results_df.groupby('config')[['diracc', 'mae', 'rmse']].mean().sort_values('diracc', ascending=False))
    
    return results_df

#### General model parameters:

In [93]:
horizon = 5
step = 10
date_start = '2015-01-01'
date_end = '2019-12-31'

In [95]:
configs = { 
    'task_1_model': {
        'n_lags': 5,
        'parkinson_vol_windows': (5, 10),
        'alpha_grid': (10.0, 50.0, 100.0)
    }
}

baseline_results = run_evaluation(
    configs,
    prices_df,
    tickers,
    date_start,
    date_end,
    horizon,
    step,
    return_predictions=True,
    verbose=True
)


____________________________________________________________
Config: task_1_model
Config: {'n_lags': 5, 'parkinson_vol_windows': (5, 10), 'alpha_grid': (10.0, 50.0, 100.0)}

XLK: DirAcc=0.5650, MAE=0.017077, RMSE=0.023033, 
XLP: DirAcc=0.4916, MAE=0.011825, RMSE=0.016281, 
XLV: DirAcc=0.5068, MAE=0.015600, RMSE=0.020888, 
XLF: DirAcc=0.5219, MAE=0.017444, RMSE=0.023977, 
XLE: DirAcc=0.4916, MAE=0.022640, RMSE=0.030074, 
XLI: DirAcc=0.5180, MAE=0.016183, RMSE=0.021902, 

Test Results:
                diracc       mae      rmse
config                                    
task_1_model  0.515829  0.016795  0.022692


Our original model using Ridge uses flat vector featuresets, but attention-based models require structured 3D tensors (batch × lags × features).